In [1]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/IDSE_PROJECT"

DATA_PATH = os.path.join(PROJECT_PATH, "processed_data")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
RESULT_PATH = os.path.join(PROJECT_PATH, "results")

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

print("Project folder connected successfully")


Mounted at /content/drive
Project folder connected successfully


In [3]:

import pandas as pd
import numpy as np

cleaned_df = pd.read_csv(
    os.path.join(DATA_PATH, "cleaned_online_retail.csv")
)

cleaned_df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [6]:
valid_sales = cleaned_df[
    cleaned_df["CustomerID"].notna()
].copy()

print("Valid customer transactions:", valid_sales.shape)

print("Number of unique customers:",
      valid_sales["CustomerID"].nunique())

Valid customer transactions: (397884, 9)
Number of unique customers: 4338


In [8]:
import pandas as pd

# Convert InvoiceDate to datetime format
valid_sales["InvoiceDate"] = pd.to_datetime(valid_sales["InvoiceDate"])

cutoff_date = pd.Timestamp("2011-09-01")

past_data = valid_sales[
    valid_sales["InvoiceDate"] < cutoff_date
].copy()

future_data = valid_sales[
    valid_sales["InvoiceDate"] >= cutoff_date
].copy()

print("Past transactions:", len(past_data))
print("Future transactions:", len(future_data))

print("Past customers:",
      past_data["CustomerID"].nunique())

print("Future customers:",
      future_data["CustomerID"].nunique())

Past transactions: 226467
Future transactions: 171417
Past customers: 3317
Future customers: 2973


In [9]:
past_data["Revenue"] = (
    past_data["Quantity"] * past_data["UnitPrice"]
)

customer_data = past_data.groupby(
    "CustomerID"
).agg(
    LastPurchase=("InvoiceDate", "max"),
    InvoiceFrequency=("InvoiceNo", "nunique"),
    TotalQuantity=("Quantity", "sum"),
    TotalRevenue=("Revenue", "sum"),
    ProductDiversity=("StockCode", "nunique"),
    ActiveDays=("InvoiceDate",
                lambda x: x.dt.date.nunique())
)

customer_data["RecencyDays"] = (
    cutoff_date - customer_data["LastPurchase"]
).dt.days

customer_data = customer_data.drop(
    columns=["LastPurchase"]
)

future_customers = future_data[
    "CustomerID"
].unique()

customer_data["Returned"] = (
    customer_data.index.isin(future_customers)
).astype(int)

print(customer_data.head())

print("\nTarget distribution:")
print(customer_data["Returned"].value_counts())

            InvoiceFrequency  TotalQuantity  TotalRevenue  ProductDiversity  \
CustomerID                                                                    
12346.0                    1          74215      77183.60                 1   
12347.0                    5           1590       2790.86                82   
12348.0                    3           2124       1487.24                22   
12350.0                    1            197        334.40                17   
12352.0                    5            254       1561.81                26   

            ActiveDays  RecencyDays  Returned  
CustomerID                                     
12346.0              1          225         0  
12347.0              5           29         1  
12348.0              3          148         1  
12350.0              1          210         0  
12352.0              4          162         1  

Target distribution:
Returned
1    1952
0    1365
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X = customer_data.drop(
    columns=["Returned"]
)

y = customer_data["Returned"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (2653, 6)
Testing data: (664, 6)


In [11]:

X_train.to_csv(os.path.join(DATA_PATH, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(DATA_PATH, "X_test.csv"), index=False)

pd.Series(y_train).to_csv(
    os.path.join(DATA_PATH, "y_train.csv"), index=False
)

pd.Series(y_test).to_csv(
    os.path.join(DATA_PATH, "y_test.csv"), index=False
)

print("Training and testing datasets saved")


Training and testing datasets saved
